# Dynamic Directory Structure Tracking for DIR.TAG

This notebook provides tools to generate a highly condensed, machine-readable tree output for tracking with DIR.TAG files. It helps maintain the directory tracking system used in the AutoGen project.

## Purpose

The AutoGen project uses DIR.TAG files to track development status and technical debt across its directory structure. This notebook helps by:

1. Generating a complete directory tree in a format suitable for DIR.TAG tracking
2. Comparing the current directory structure with the tracked structure in DIR.TAG files
3. Identifying directories that need new DIR.TAG files
4. Producing a machine-readable format for automation tools

## Required Libraries

Let's start by importing the necessary libraries:

In [ ]:
import os
import re
import glob
import json
from pathlib import Path
import xml.etree.ElementTree as ET
from xml.dom import minidom
from datetime import datetime

## Configuration

Set up the configuration for the directory structure tracking:

In [ ]:
# Base project directory
BASE_DIR = r"c:\Projects\autogen"

# Directories to exclude from tracking
EXCLUDE_DIRS = [
    '.git',
    '.vs',
    'node_modules',
    '__pycache__',
    '.venv',
    'bin',
    'obj',
    '.vscode-server',
    'artifacts'
]

# File patterns to exclude
EXCLUDE_FILE_PATTERNS = [
    '*.pyc',
    '*.pyo',
    '*.dll',
    '*.exe',
    '*.pdb',
    '*.cache'
]

# DIR.TAG configuration file location
DIR_TAG_CONFIG = os.path.join(BASE_DIR, '.config', 'dir-tag', 'dir-tag-config.xml')

## Functions for Directory Structure Analysis

In [ ]:
def should_exclude_path(path, relative_to_base=True):
    """Determine if a path should be excluded from tracking."""
    if relative_to_base:
        path = os.path.relpath(path, BASE_DIR)
    
    path_parts = path.split(os.sep)
    
    # Check if any part of path matches an excluded directory
    for part in path_parts:
        if part in EXCLUDE_DIRS:
            return True
    
    # Check if the path is a file that matches an excluded pattern
    if os.path.isfile(os.path.join(BASE_DIR, path)):
        for pattern in EXCLUDE_FILE_PATTERNS:
            if glob.fnmatch.fnmatch(path, pattern):
                return True
    
    return False

def generate_directory_tree(start_path=BASE_DIR, depth=0, max_depth=None, format_type='condensed'):
    """Generate a directory tree structure in the specified format.
    
    Args:
        start_path: The starting directory path
        depth: Current recursion depth
        max_depth: Maximum recursion depth (None for unlimited)
        format_type: Output format - 'condensed', 'full', or 'json'
        
    Returns:
        A list of paths in the specified format
    """
    if max_depth is not None and depth > max_depth:
        return []
    
    if should_exclude_path(start_path, relative_to_base=False):
        return []
    
    result = []
    rel_path = os.path.relpath(start_path, BASE_DIR)
    if rel_path != '.':
        if format_type == 'condensed':
            result.append(rel_path.replace('\\', '/'))
        elif format_type == 'full':
            indent = '  ' * depth
            result.append(f"{indent}{os.path.basename(start_path)}/")
        elif format_type == 'json':
            result.append({
                'path': rel_path.replace('\\', '/'),
                'type': 'directory',
                'depth': depth,
                'has_dir_tag': os.path.exists(os.path.join(start_path, 'DIR.TAG'))
            })
    
    try:
        entries = sorted(os.listdir(start_path))
        dirs = []
        files = []
        
        for entry in entries:
            full_path = os.path.join(start_path, entry)
            rel_entry_path = os.path.relpath(full_path, BASE_DIR)
            
            if should_exclude_path(full_path, relative_to_base=False):
                continue
                
            if os.path.isdir(full_path):
                dirs.append(entry)
            else:
                if format_type == 'condensed':
                    files.append(rel_entry_path.replace('\\', '/'))
                elif format_type == 'full':
                    indent = '  ' * (depth + 1)
                    files.append(f"{indent}{entry}")
                elif format_type == 'json':
                    files.append({
                        'path': rel_entry_path.replace('\\', '/'),
                        'type': 'file',
                        'depth': depth + 1,
                        'is_dir_tag': entry == 'DIR.TAG'
                    })
        
        # Add files first
        result.extend(files)
        
        # Then recursively add directories
        for dir_entry in dirs:
            result.extend(generate_directory_tree(
                os.path.join(start_path, dir_entry),
                depth + 1,
                max_depth,
                format_type
            ))
            
        return result
    except PermissionError:
        # Handle permission errors gracefully
        if format_type == 'json':
            return [{'path': rel_path.replace('\\', '/'), 'error': 'Permission denied'}]
        return [f"ERROR: Permission denied for {rel_path}"]

## Functions for DIR.TAG File Management

In [ ]:
def find_dir_tag_files():
    """Find all DIR.TAG files in the project."""
    dir_tags = []
    for root, dirs, files in os.walk(BASE_DIR):
        # Skip excluded directories
        dirs[:] = [d for d in dirs if d not in EXCLUDE_DIRS]
        
        if 'DIR.TAG' in files:
            rel_path = os.path.relpath(root, BASE_DIR)
            if rel_path == '.':
                rel_path = ''
            dir_tags.append(rel_path.replace('\\', '/'))
    
    return sorted(dir_tags)

def parse_dir_tag_file(file_path):
    """Parse a DIR.TAG file and extract its contents."""
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            content = f.read()
        
        data = {}
        
        # Extract INDEX
        index_match = re.search(r'#INDEX:\s*(.*)', content)
        if index_match:
            data['index'] = index_match.group(1).strip()
        
        # Extract status
        status_match = re.search(r'status:\s*(.*)', content)
        if status_match:
            data['status'] = status_match.group(1).strip()
        
        # Extract TODO items
        todo_items = []
        todo_section = re.search(r'#TODO:(.*?)(?:^[^#\s]|\Z)', content, re.DOTALL | re.MULTILINE)
        if todo_section:
            todo_text = todo_section.group(1).strip()
            for line in todo_text.split('\n'):
                line = line.strip()
                if line.startswith('-'):
                    todo_items.append(line[1:].strip())
        
        data['todo_items'] = todo_items
        
        # Extract updated timestamp
        updated_match = re.search(r'updated:\s*(.*)', content)
        if updated_match:
            data['updated'] = updated_match.group(1).strip()
        
        # Extract description
        desc_match = re.search(r'description:\s*\|\s*([\s\S]*?)(?:^[^\s]|\Z)', content, re.MULTILINE)
        if desc_match:
            data['description'] = desc_match.group(1).strip()
        
        return data
    except Exception as e:
        return {'error': str(e)}

def analyze_dir_tag_coverage():
    """Analyze the coverage of DIR.TAG files across the project."""
    # Get all directories
    all_dirs = [item for item in generate_directory_tree(format_type='condensed') 
                if os.path.isdir(os.path.join(BASE_DIR, item))]
    
    # Get directories with DIR.TAG files
    dirs_with_tag = find_dir_tag_files()
    
    # Find directories without DIR.TAG files
    dirs_without_tag = [d for d in all_dirs if d not in dirs_with_tag]
    
    return {
        'total_directories': len(all_dirs),
        'directories_with_dir_tag': len(dirs_with_tag),
        'directories_without_dir_tag': len(dirs_without_tag),
        'coverage_percentage': round(len(dirs_with_tag) / len(all_dirs) * 100, 2) if all_dirs else 0,
        'dirs_with_tag': dirs_with_tag,
        'dirs_without_tag': dirs_without_tag
    }

## Functions for XML Configuration Generation

In [ ]:
def generate_dir_tag_xml_config(directories):
    """Generate an XML configuration file for DIR.TAG tracking."""
    root = ET.Element("DirectoryTagConfiguration")
    
    metadata = ET.SubElement(root, "Metadata")
    ET.SubElement(metadata, "GeneratedTimestamp").text = datetime.now().isoformat()
    ET.SubElement(metadata, "TotalTrackedDirectories").text = str(len(directories))
    
    directories_el = ET.SubElement(root, "TrackedDirectories")
    
    for directory in sorted(directories):
        dir_el = ET.SubElement(directories_el, "Directory")
        ET.SubElement(dir_el, "Path").text = directory
        dir_tag_path = os.path.join(BASE_DIR, directory, 'DIR.TAG')
        
        if os.path.exists(dir_tag_path):
            dir_tag_data = parse_dir_tag_file(dir_tag_path)
            status_el = ET.SubElement(dir_el, "Status")
            status_el.text = dir_tag_data.get('status', 'UNKNOWN')
            ET.SubElement(dir_el, "LastUpdated").text = dir_tag_data.get('updated', '')
        else:
            ET.SubElement(dir_el, "Status").text = "NOT_TRACKED"
    
    # Format the XML with proper indentation
    xml_string = ET.tostring(root, encoding='utf-8')
    parsed_xml = minidom.parseString(xml_string)
    pretty_xml = parsed_xml.toprettyxml(indent="  ")
    
    return pretty_xml

def save_dir_tag_xml_config(xml_content, file_path=None):
    """Save the DIR.TAG XML configuration to a file."""
    if file_path is None:
        # Default location in the .config/dir-tag directory
        file_path = os.path.join(BASE_DIR, '.config', 'dir-tag', 'dir-tag-tracking.xml')
    
    # Ensure the directory exists
    os.makedirs(os.path.dirname(file_path), exist_ok=True)
    
    with open(file_path, 'w', encoding='utf-8') as f:
        f.write(xml_content)
    
    return file_path

## Generate Condensed Directory Tree

This format produces a machine-readable, highly condensed tree structure suitable for DIR.TAG tracking:

In [ ]:
condensed_tree = generate_directory_tree(format_type='condensed')

# Print a sample of the tree (first 20 entries)
print("\n".join(condensed_tree[:20]))
print(f"\n... and {len(condensed_tree) - 20} more entries")

## Generate Machine-Readable JSON Structure

This more detailed format includes metadata about each directory and file:

In [ ]:
json_tree = generate_directory_tree(format_type='json')

# Print a sample of the tree in JSON format
import json
print(json.dumps(json_tree[:5], indent=2))

## Analyze DIR.TAG Coverage

This analysis shows which directories have DIR.TAG files and which ones are missing them:

In [ ]:
coverage = analyze_dir_tag_coverage()

print(f"Total directories: {coverage['total_directories']}")
print(f"Directories with DIR.TAG: {coverage['directories_with_dir_tag']}")
print(f"Directories without DIR.TAG: {coverage['directories_without_dir_tag']}")
print(f"Coverage percentage: {coverage['coverage_percentage']}%")

# Show sample of directories missing DIR.TAG files
print("\nSample of directories missing DIR.TAG files:")
for d in coverage['dirs_without_tag'][:10]:
    print(f"  - {d}")
if len(coverage['dirs_without_tag']) > 10:
    print(f"  ... and {len(coverage['dirs_without_tag']) - 10} more")

## Generate XML Configuration for DIR.TAG Tracking

This generates a standardized XML file that can be used by automation tools:

In [ ]:
# Get all directories
all_dirs = [item for item in condensed_tree if os.path.isdir(os.path.join(BASE_DIR, item))]

# Generate XML configuration
xml_config = generate_dir_tag_xml_config(all_dirs)

# Print a preview of the XML
print(xml_config[:500] + "\n...")

## Save XML Configuration

Save the generated XML configuration to a file for use with automation tools:

In [ ]:
output_file = save_dir_tag_xml_config(xml_config)
print(f"XML configuration saved to: {output_file}")

## Export Directory Tree for DIR.TAG Integration

Create a simple text file with the directory tree in the format needed for DIR.TAG tracking:

In [ ]:
def export_tree_for_dirtag(tree_data, output_file=None):
    """Export the directory tree in a format suitable for DIR.TAG tracking."""
    if output_file is None:
        output_file = os.path.join(BASE_DIR, '.config', 'dir-tag', 'directory-tree.txt')
    
    # Ensure the directory exists
    os.makedirs(os.path.dirname(output_file), exist_ok=True)
    
    with open(output_file, 'w', encoding='utf-8') as f:
        for item in tree_data:
            if os.path.isdir(os.path.join(BASE_DIR, item)):
                f.write(f"{item}\n")
    
    return output_file

# Export the tree
export_file = export_tree_for_dirtag(condensed_tree)
print(f"Directory tree exported to: {export_file}")

## Utility Function to Generate a New DIR.TAG File

This function can be used to generate a new DIR.TAG file for a directory that doesn't have one:

In [ ]:
def generate_new_dirtag(directory_path, description="", status="NOT_STARTED"):
    """Generate a new DIR.TAG file for the specified directory."""
    if not os.path.isdir(directory_path):
        return f"Error: {directory_path} is not a valid directory"
    
    # Get relative path from BASE_DIR
    rel_path = os.path.relpath(directory_path, BASE_DIR).replace('\\', '/')
    if rel_path == '.':
        rel_path = ''
    
    dirtag_content = f"""#INDEX: {rel_path}
#TODO:
  - Update this DIR.TAG file with appropriate tasks NOT_STARTED
status: {status}
updated: {datetime.now().strftime('%Y-%m-%dT%H:%M:%SZ')}
description: |
  {description}
"""
    
    output_path = os.path.join(directory_path, "DIR.TAG")
    
    # Don't overwrite existing DIR.TAG files
    if os.path.exists(output_path):
        return f"Error: DIR.TAG already exists in {directory_path}"
    
    with open(output_path, 'w', encoding='utf-8') as f:
        f.write(dirtag_content)
    
    return f"Created DIR.TAG in {directory_path}"

# Example: Generate a DIR.TAG file for a directory
# Uncomment and modify to use
# target_dir = os.path.join(BASE_DIR, 'some', 'directory')
# result = generate_new_dirtag(target_dir, "Description of the directory's purpose")
# print(result)

## Conclusion

This notebook provides tools to:

1. Generate a machine-readable directory tree for DIR.TAG tracking
2. Analyze coverage of DIR.TAG files across the project
3. Generate XML configuration for automation tools
4. Create new DIR.TAG files where needed

The output files can be integrated with the toolbox modules and PowerShell scripts that manage the DIR.TAG system in the AutoGen project.